# Guia de Modelos Pre-entrenados de Deep Learning

Bienvenidos al semillero de **Redes Neuronales**. En este cuaderno exploraremos la increible variedad de modelos pre-entrenados que existen hoy en dia, listos para usar con apenas unas lineas de codigo. Cubriremos **vision por computador, procesamiento de lenguaje natural, audio y modelos multimodales**.

> **Que es un modelo pre-entrenado?** Es un modelo que ya fue entrenado por alguien mas (la parte costosa) y lo podemos usar directamente. Podemos pensarlo como una *funcion* que recibe un input y devuelve un output.

---

### Tabla de Contenido

| # | Seccion | Tipo |
|---|---------|------|
| 1 | Clasificacion de Imagenes (ResNet) | Vision |
| 2 | Deteccion de Objetos (DETR) | Vision |
| 3 | Caballos a Cebras (CycleGAN) | Vision / Generativo |
| 4 | Segmentacion de Imagenes (DeepLabV3 + Mask R-CNN) | Vision |
| 5 | Estimacion de Profundidad (DPT) | Vision |
| 6 | Clasificacion Zero-Shot (CLIP) | Multimodal |
| 7 | Descripcion de Imagenes (BLIP) | Multimodal |
| 8 | Generacion de Texto (GPT-2) | NLP |
| 9 | Analisis de Sentimiento | NLP |
| 10 | Resumen Automatico (BART) | NLP |
| 11 | Traduccion Automatica (MarianMT) | NLP |
| 12 | Preguntas y Respuestas (QA) | NLP |
| 13 | Reconocimiento de Entidades (NER) | NLP |
| 14 | Fill-Mask con BERT | NLP |
| 15 | Transcripcion de Voz (Whisper) | Audio |
| 16 | Clasificacion de Audio | Audio |
| 17 | Generacion de Musica (MusicGen) | Audio |
| 18 | Generacion de Imagenes (Stable Diffusion) | Generativo |
| 19 | Visual Question Answering (BLIP-VQA) | Multimodal |

---

**IMPORTANTE:**
1. Asegurarse de usar **GPU** en Colab: *Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > T4 GPU*
2. Subir los archivos necesarios (imagenes, `clases.txt`, `horse2zebra_0.4.0.pth`) al espacio de ficheros de Colab

## 0. Instalacion de dependencias

Ejecuta esta celda una sola vez al inicio.

In [ ]:
!pip install torch torchvision torchaudio --quiet
!pip install transformers diffusers accelerate safetensors soundfile datasets --quiet
!pip install Pillow matplotlib numpy --quiet

### Imports comunes y utilidades

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Audio

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

def liberar_memoria():
    import gc; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print("Memoria GPU liberada.")

---
# 1. Clasificacion de Imagenes con ResNet

El modelo **ResNet101** fue entrenado en **ImageNet** (1000 categorias, millones de imagenes). Le damos una imagen y nos dice que hay en ella.

**Como funciona:**
- La imagen se redimensiona y normaliza para coincidir con el formato de entrenamiento
- Pasa por ~175 millones de parametros organizados en capas convolucionales
- Produce un vector de 1000 scores, uno por categoria
- La categoria con el score mas alto es la prediccion

In [ ]:
from torchvision import models, transforms
from PIL import Image

resnet = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
resnet.eval()
print(f"Parametros: {sum(p.numel() for p in resnet.parameters()):,}")

### Pipeline de preprocesamiento

Antes de pasarle una imagen al modelo, debemos transformarla:
1. **Resize** a 256x256
2. **CenterCrop** a 224x224
3. **ToTensor** - convierte a tensor de PyTorch
4. **Normalize** - normaliza RGB para coincidir con el entrenamiento

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

### Inferencia sobre una imagen

Sube una imagen al Colab. El proceso de correr un modelo entrenado en datos nuevos se llama **inferencia**.

In [ ]:
# Cambia el nombre del archivo por tu imagen
img = Image.open("linda_bella.jpg")
display(img)

img_t = preprocess(img)
batch_t = torch.unsqueeze(img_t, 0)
out = resnet(batch_t)

with open('clases.txt') as f:
    labels = [line.strip() for line in f.readlines()]

percentage = torch.nn.functional.softmax(out, dim=1)[0] * 100
_, indices = torch.sort(out, descending=True)

print("Top 5 predicciones:")
for idx in indices[0][:5]:
    print(f"  {labels[idx]:>50s} -> {percentage[idx].item():.2f}%")

In [ ]:
# Probemos con otra imagen
img = Image.open("tralalero.jpg").convert("RGB")
display(img)

img_t = preprocess(img)
batch_t = torch.unsqueeze(img_t, 0)
out = resnet(batch_t)
percentage = torch.nn.functional.softmax(out, dim=1)[0] * 100
_, indices = torch.sort(out, descending=True)

print("Top 5 predicciones:")
for idx in indices[0][:5]:
    print(f"  {labels[idx]:>50s} -> {percentage[idx].item():.2f}%")

In [ ]:
liberar_memoria()

---
# 2. Deteccion de Objetos con DETR

La **deteccion de objetos** va mas alla de la clasificacion: no solo dice *que* hay en la imagen, sino *donde* esta, dibujando **bounding boxes** alrededor de cada objeto.

**DETR** (DEtection TRansformer) de Facebook AI combina una CNN con Transformers para detectar objetos.

**Modelo:** `facebook/detr-resnet-50` (entrenado en COCO, 91 categorias)

In [ ]:
from transformers import DetrImageProcessor, DetrForObjectDetection
from PIL import ImageDraw
import random

processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
detr_model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")
detr_model.eval()

img = Image.open("copito.jpg").convert("RGB")

inputs = processor(images=img, return_tensors="pt")
with torch.no_grad():
    outputs = detr_model(**inputs)

target_sizes = torch.tensor([img.size[::-1]])
results = processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=0.7)[0]

draw = ImageDraw.Draw(img)
for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
    box = [round(i, 2) for i in box.tolist()]
    color = (random.randint(50,255), random.randint(50,255), random.randint(50,255))
    draw.rectangle(box, outline=color, width=3)
    text = f"{detr_model.config.id2label[label.item()]}: {score:.2f}"
    draw.text((box[0], box[1]-15), text, fill=color)

display(img)
print(f"Objetos detectados: {len(results['scores'])}")
for score, label in zip(results["scores"], results["labels"]):
    print(f"  {detr_model.config.id2label[label.item()]:>20s} -> {score:.2f}")

In [ ]:
del detr_model, processor
liberar_memoria()

---
# 3. Caballos a Cebras con CycleGAN

Usaremos una **Red Generativa Adversarial** (GAN) para transformar imagenes de caballos en cebras.

### Que es una GAN?
Imaginemos que queremos vender pinturas falsas de artistas famosos. Contratamos un estudiante de arte para juzgar nuestras falsificaciones. El mejora detectando falsas, y nosotros mejoramos creandolas. Al final, nuestras pinturas son indistinguibles de las reales.

- **Red Generadora**: crea imagenes falsas
- **Red Discriminadora**: distingue falsas de reales
- Ambas mejoran en un loop adversarial

**CycleGAN** permite traducir imagenes de un dominio a otro sin pares alineados.

> Necesitas subir `horse2zebra_0.4.0.pth` al Colab.

In [ ]:
import torch.nn as nn

class ResnetBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.ReflectionPad2d(1), nn.Conv2d(dim, dim, 3, bias=False),
            nn.InstanceNorm2d(dim, affine=False), nn.ReLU(True),
            nn.ReflectionPad2d(1), nn.Conv2d(dim, dim, 3, bias=False),
            nn.InstanceNorm2d(dim, affine=False))
    def forward(self, x): return x + self.conv_block(x)

class ResnetGenerator(nn.Module):
    def __init__(self, input_nc=3, output_nc=3, ngf=64, n_blocks=9):
        super().__init__()
        model = [nn.ReflectionPad2d(3), nn.Conv2d(input_nc, ngf, 7, bias=False),
                 nn.InstanceNorm2d(ngf, affine=False), nn.ReLU(True)]
        for i in range(2):
            mult = 2**i
            model += [nn.Conv2d(ngf*mult, ngf*mult*2, 3, stride=2, padding=1, bias=False),
                      nn.InstanceNorm2d(ngf*mult*2, affine=False), nn.ReLU(True)]
        for _ in range(n_blocks): model += [ResnetBlock(ngf*4)]
        for i in range(2):
            mult = 2**(2-i)
            model += [nn.ConvTranspose2d(ngf*mult, ngf*mult//2, 3, stride=2, padding=1, output_padding=1, bias=False),
                      nn.InstanceNorm2d(ngf*mult//2, affine=False), nn.ReLU(True)]
        model += [nn.ReflectionPad2d(3), nn.Conv2d(ngf, output_nc, 7), nn.Tanh()]
        self.model = nn.Sequential(*model)
    def forward(self, x): return self.model(x)

netG = ResnetGenerator()
netG.load_state_dict(torch.load("horse2zebra_0.4.0.pth", map_location="cpu"), strict=False)
netG.eval()
print("CycleGAN cargada.")

In [ ]:
preprocess_gan = transforms.Compose([transforms.Resize(256), transforms.ToTensor()])

img = Image.open("caballo2.jpg")
print("Original:")
display(img)

img_t = preprocess_gan(img)
batch_out = netG(img_t.unsqueeze(0))
out_img = transforms.ToPILImage()((batch_out.data.squeeze() + 1.0) / 2.0)
print("Caballo -> Cebra:")
display(out_img)

In [ ]:
liberar_memoria()

---
# 4. Segmentacion de Imagenes

La **segmentacion** clasifica *cada pixel* de una imagen:

| Tipo | Descripcion | Modelo |
|------|-------------|--------|
| **Semantica** | Cada pixel recibe una clase | DeepLabV3 |
| **De Instancias** | Diferencia objetos individuales | Mask R-CNN |

**Aplicaciones:** Conduccion autonoma, imagenes medicas, edicion de fotos.

In [ ]:
import torchvision

img = Image.open("copito.jpg")
preprocess_seg = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
input_tensor = preprocess_seg(img).unsqueeze(0)

# DeepLabV3
deeplab = torchvision.models.segmentation.deeplabv3_resnet101(weights="DEFAULT")
deeplab.eval()
with torch.no_grad():
    output = deeplab(input_tensor)["out"][0]
segmentation = output.argmax(0).byte().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img); axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(img); axes[1].imshow(segmentation, alpha=0.5, cmap="jet")
axes[1].set_title("DeepLabV3 - Segmentacion Semantica"); axes[1].axis("off")
plt.tight_layout(); plt.show()

# Mask R-CNN
maskrcnn = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")
maskrcnn.eval()
with torch.no_grad():
    predictions = maskrcnn([transforms.ToTensor()(img)])

plt.figure(figsize=(10, 8))
plt.imshow(img)
for mask, score, label in zip(predictions[0]['masks'], predictions[0]['scores'], predictions[0]['labels']):
    if score > 0.5:
        plt.imshow(mask[0].cpu().numpy(), alpha=0.4, cmap="spring")
        print(f"  Detectado: label={label.item()} ({score:.2f})")
plt.title("Mask R-CNN - Segmentacion de Instancias"); plt.axis("off"); plt.show()

In [ ]:
del deeplab, maskrcnn
liberar_memoria()

---
# 5. Estimacion de Profundidad (Depth Estimation)

Los modelos de **estimacion de profundidad** predicen la distancia de cada pixel al observador, creando un **mapa de profundidad** a partir de una sola imagen 2D.

**Aplicaciones:** Realidad aumentada, conduccion autonoma, reconstruccion 3D.

**Modelo:** `Intel/dpt-large` (Dense Prediction Transformer)

In [ ]:
from transformers import DPTImageProcessor, DPTForDepthEstimation

dpt_processor = DPTImageProcessor.from_pretrained("Intel/dpt-large")
dpt_model = DPTForDepthEstimation.from_pretrained("Intel/dpt-large")
dpt_model.eval()

img = Image.open("copito.jpg").convert("RGB")

inputs = dpt_processor(images=img, return_tensors="pt")
with torch.no_grad():
    predicted_depth = dpt_model(**inputs).predicted_depth

prediction = torch.nn.functional.interpolate(
    predicted_depth.unsqueeze(1), size=img.size[::-1], mode="bicubic", align_corners=False
).squeeze()
depth_map = prediction.cpu().numpy()
depth_map = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min())

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img); axes[0].set_title("Imagen Original"); axes[0].axis("off")
im = axes[1].imshow(depth_map, cmap="inferno")
axes[1].set_title("Mapa de Profundidad (DPT)"); axes[1].axis("off")
plt.colorbar(im, ax=axes[1], fraction=0.046, label="Cerca <-> Lejos")
plt.tight_layout(); plt.show()

In [ ]:
del dpt_model, dpt_processor
liberar_memoria()

---
# 6. Clasificacion Zero-Shot con CLIP

**CLIP** (Contrastive Language-Image Pre-training) de OpenAI conecta **imagenes y texto** en un mismo espacio. Puede clasificar imagenes en categorias que **nunca vio durante el entrenamiento**.

**Como funciona:**
1. Toma una imagen y una lista de descripciones textuales
2. Calcula la similitud entre la imagen y cada descripcion
3. La descripcion mas similar es la prediccion

Podemos inventar las categorias que queramos!

In [ ]:
from transformers import CLIPProcessor, CLIPModel

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

img = Image.open("linda_bella.jpg").convert("RGB")
display(img)

candidate_labels = [
    "a photo of a cat", "a photo of a dog", "a photo of a bird",
    "a photo of a fish", "a photo of a horse", "a photo of a person",
]

inputs = clip_processor(text=candidate_labels, images=img, return_tensors="pt", padding=True)
with torch.no_grad():
    outputs = clip_model(**inputs)

probs = outputs.logits_per_image.softmax(dim=1)[0]

print("Resultados CLIP (Zero-Shot):")
for label, prob in sorted(zip(candidate_labels, probs), key=lambda x: x[1], reverse=True):
    bar = "#" * int(prob * 40)
    print(f"  {label:>30s}: {prob:.1%} {bar}")

In [ ]:
del clip_model, clip_processor
liberar_memoria()

---
# 7. Descripcion Automatica de Imagenes con BLIP

**BLIP** genera descripciones en lenguaje natural de lo que ve en una imagen (Image Captioning).

**Modelo:** `Salesforce/blip-image-captioning-base`

**Aplicaciones:** Accesibilidad, organizacion de fotos, motores de busqueda.

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model.eval()

for img_name in ["linda_bella.jpg", "copito.jpg"]:
    try:
        img = Image.open(img_name).convert("RGB")
        display(img)
        inputs = blip_processor(img, return_tensors="pt")
        with torch.no_grad():
            out = blip_model.generate(**inputs, max_new_tokens=50)
        print(f"  Caption: {blip_processor.decode(out[0], skip_special_tokens=True)}")
        print()
    except FileNotFoundError:
        print(f"  ({img_name} no encontrado)")

In [ ]:
del blip_model, blip_processor
liberar_memoria()

---
# 8. Generacion de Texto con GPT-2

Entramos al **Procesamiento del Lenguaje Natural** (NLP). **DistilGPT-2** completa texto a partir de un prompt, prediciendo la siguiente palabra mas probable token por token.

GPT-2 (2019) es el abuelo de modelos como ChatGPT.

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("distilgpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("distilgpt2")
gpt2_model.eval()

def generar_texto(prompt, max_length=80, temperature=0.7):
    inputs = tokenizer.encode(prompt, return_tensors="pt")
    outputs = gpt2_model.generate(
        inputs, attention_mask=torch.ones_like(inputs),
        max_length=max_length, do_sample=True, top_k=50,
        temperature=temperature, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

for p in ["The future of artificial intelligence is",
          "Once upon a time in a galaxy far away",
          "The most important thing about deep learning is"]:
    print(f"Prompt: {p}")
    print(f"  -> {generar_texto(p)}")
    print("-" * 60)

In [ ]:
del gpt2_model, tokenizer
liberar_memoria()

---
# 9. Analisis de Sentimiento

Determina si un texto expresa una opinion **positiva** o **negativa**. Una de las aplicaciones mas comunes de NLP.

**Aplicaciones:** Redes sociales, resenas de productos, atencion al cliente.

In [ ]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis")

textos = [
    "I absolutely love this movie, it was fantastic!",
    "This product is terrible, I want my money back.",
    "The weather today is okay, nothing special.",
    "Best purchase I have ever made, highly recommended!",
    "I'm so disappointed with the service, never coming back.",
]

print("Analisis de Sentimiento:")
print("=" * 60)
for texto in textos:
    r = sentiment(texto)[0]
    emoji = "+" if r["label"] == "POSITIVE" else "-"
    print(f"  [{emoji}] {r['label']:>8s} ({r['score']:.2%}): {texto}")

In [ ]:
liberar_memoria()

---
# 10. Resumen Automatico de Texto con BART

**BART** condensa textos largos manteniendo la informacion clave.

**Modelo:** `facebook/bart-large-cnn` (afinado en articulos de CNN/DailyMail)

In [ ]:
from transformers import pipeline

summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

articulo = '''
Artificial intelligence has made tremendous strides in recent years, transforming
industries from healthcare to transportation. Machine learning algorithms can now
diagnose diseases from medical images with accuracy rivaling human doctors. In the
automotive industry, self-driving cars are being tested on public roads. Natural
language processing has enabled chatbots and virtual assistants that understand human
queries with remarkable fluency. However, these advances raise important ethical
questions about privacy, job displacement, and the potential misuse of AI technology.
Researchers and policymakers are working to establish guidelines that ensure AI
development benefits society while minimizing risks.
'''

resumen = summarizer(articulo, max_length=60, min_length=20, do_sample=False)

print("TEXTO ORIGINAL:")
print(articulo.strip())
print("\n" + "=" * 60)
print("\nRESUMEN GENERADO:")
print(resumen[0]["summary_text"])

In [ ]:
liberar_memoria()

---
# 11. Traduccion Automatica con MarianMT

Los modelos **MarianMT** de Helsinki-NLP son traductores especializados para pares de idiomas. Rapidos y precisos.

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

def traducir(texto, src, tgt):
    name = f"Helsinki-NLP/opus-mt-{src}-{tgt}"
    tok = MarianTokenizer.from_pretrained(name)
    mdl = MarianMTModel.from_pretrained(name); mdl.eval()
    inputs = tok(texto, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        translated = mdl.generate(**inputs)
    result = tok.decode(translated[0], skip_special_tokens=True)
    del mdl, tok
    return result

texto_es = "Las redes neuronales son una herramienta poderosa para resolver problemas complejos."
print(f"ES: {texto_es}")
print(f"EN: {traducir(texto_es, 'es', 'en')}")
print()

texto_en = "Deep learning is transforming the world of artificial intelligence."
print(f"EN: {texto_en}")
print(f"FR: {traducir(texto_en, 'en', 'fr')}")
print()
print(f"EN: {texto_en}")
print(f"DE: {traducir(texto_en, 'en', 'de')}")

In [ ]:
liberar_memoria()

---
# 12. Preguntas y Respuestas (Question Answering)

Un modelo de **QA extractivo** recibe un **contexto** y una **pregunta**, y extrae la respuesta directamente del texto.

**Modelo:** `distilbert-base-cased-distilled-squad`

In [ ]:
from transformers import pipeline

qa = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

contexto = '''
PyTorch is an open-source machine learning framework developed by Facebook AI Research lab.
It was released in 2016 and has become one of the most popular frameworks for deep learning,
alongside TensorFlow by Google. PyTorch is known for its dynamic computational graph,
making it easier to debug. It is written in Python and C++, and supports GPU acceleration.
'''

for pregunta in ["Who developed PyTorch?", "When was PyTorch released?",
                 "What is PyTorch known for?", "What languages is PyTorch written in?"]:
    r = qa(question=pregunta, context=contexto)
    print(f"  P: {pregunta}")
    print(f"  R: {r['answer']} ({r['score']:.2%})\n")

In [ ]:
liberar_memoria()

---
# 13. Reconocimiento de Entidades Nombradas (NER)

**NER** identifica y clasifica entidades en texto: **personas (PER)**, **organizaciones (ORG)**, **lugares (LOC)**, etc.

**Aplicaciones:** Extraccion de informacion, analisis de documentos, chatbots.

In [ ]:
from transformers import pipeline

ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

textos = [
    "Elon Musk founded SpaceX in Hawthorne, California in 2002.",
    "The Eiffel Tower in Paris was built by Gustave Eiffel for the 1889 World Fair.",
    "Gabriel Garcia Marquez wrote One Hundred Years of Solitude in Colombia.",
]

for texto in textos:
    print(f"Texto: {texto}")
    for ent in ner(texto):
        print(f"  -> {ent['word']:>25s}  [{ent['entity_group']:>5s}]  ({ent['score']:.2f})")
    print()

In [ ]:
liberar_memoria()

---
# 14. Fill-Mask con BERT

**BERT** fue entrenado con **Masked Language Modeling**: se oculta una palabra y el modelo predice cual es. Esto demuestra que BERT entiende el contexto de las palabras.

In [ ]:
from transformers import pipeline

fill_mask = pipeline("fill-mask", model="bert-base-uncased")

frases = [
    "The capital of France is [MASK].",
    "Deep learning is a subset of [MASK] learning.",
    "Albert Einstein was a famous [MASK].",
    "Python is a popular programming [MASK].",
]

for frase in frases:
    print(f"Frase: {frase}")
    for r in fill_mask(frase)[:3]:
        print(f"  -> {r['token_str']:>15s}  ({r['score']:.2%})")
    print()

In [ ]:
liberar_memoria()

---
# 15. Transcripcion de Voz con Whisper

**Whisper** de OpenAI transcribe audio en multiples idiomas con alta precision.

**Modelo:** `openai/whisper-small`

> Al ejecutar, el microfono se activa por 5 segundos. Habla claro!

In [ ]:
from transformers import pipeline
from IPython.display import Javascript
from google.colab import output
import base64
import soundfile as sf

asr = pipeline("automatic-speech-recognition", model="openai/whisper-small")

RECORD = '''
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(reader.result.split(",")[1])
  reader.readAsDataURL(blob)
})
async function record(sec){
  const stream = await navigator.mediaDevices.getUserMedia({audio:true});
  const recorder = new MediaRecorder(stream);
  let data = [];
  recorder.ondataavailable = event => data.push(event.data);
  recorder.start();
  await sleep(sec*1000);
  recorder.stop();
  await new Promise(resolve => recorder.onstop = resolve);
  stream.getTracks().forEach(track => track.stop());
  const blob = new Blob(data);
  return await b2text(blob);
}
'''

def record(sec=5, filename="recording.wav"):
    display(Javascript(RECORD))
    audio_b64 = output.eval_js(f'record({sec})')
    with open(filename, "wb") as f:
        f.write(base64.b64decode(audio_b64))
    return filename

print("Grabando por 5 segundos... habla ahora!")
audio_path = record(5)
print("\nTranscripcion:")
print(asr(audio_path)["text"])

In [ ]:
liberar_memoria()

---
# 16. Clasificacion de Audio

Identifica lo que se escucha en una grabacion: musica, habla, ruido ambiental, instrumentos, etc.

**Modelo:** `MIT/ast-finetuned-audioset-10-10-0.4593` (Audio Spectrogram Transformer, 500+ categorias)

> Usa la grabacion de la seccion anterior.

In [ ]:
from transformers import pipeline

audio_classifier = pipeline("audio-classification", model="MIT/ast-finetuned-audioset-10-10-0.4593")

try:
    results = audio_classifier("recording.wav")
    print("Clasificacion del audio:")
    for r in results[:5]:
        bar = "#" * int(r["score"] * 40)
        print(f"  {r['label']:>30s}: {r['score']:.2%} {bar}")
except Exception as e:
    print(f"Error: {e}\nEjecuta la celda de Whisper primero para grabar audio.")

In [ ]:
liberar_memoria()

---
# 17. Generacion de Musica con MusicGen

**MusicGen** de Meta genera musica a partir de una descripcion textual, token por token (similar a GPT pero para audio).

**Modelo:** `facebook/musicgen-small`

> La generacion toma 30s-2min.

In [ ]:
from transformers import pipeline
from IPython.display import Audio, display
import soundfile as sf

generator = pipeline("text-to-audio", model="facebook/musicgen-small")

prompt = "Jazz fusion with a piano solo"
print(f"Prompt: {prompt}")
print("Generando musica...")

out = generator(prompt, generate_kwargs={"max_length": 256})
audio_arr = np.squeeze(out["audio"]) if out["audio"].ndim == 3 else out["audio"]
sf.write("musicgen_demo.wav", audio_arr, out["sampling_rate"])
display(Audio("musicgen_demo.wav"))

In [ ]:
liberar_memoria()

---
# 18. Generacion de Imagenes con Stable Diffusion

**Stable Diffusion** genera imagenes desde texto mediante **difusion**:
1. Comienza con ruido aleatorio
2. Elimina ruido paso a paso, guiado por el texto
3. Tras ~50 pasos, emerge una imagen coherente

**Modelo:** `CompVis/stable-diffusion-v1-4`

> Primera ejecucion descarga ~4GB. Generacion: ~30s con GPU.

In [ ]:
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4", torch_dtype=torch.float16)
pipe = pipe.to("cuda" if torch.cuda.is_available() else "cpu")

prompt = "A cyberpunk cityscape at night with neon lights and flying cars, digital art"
print(f"Prompt: {prompt}")

with torch.autocast("cuda"):
    image = pipe(prompt, guidance_scale=7.5).images[0]
display(image)

In [ ]:
# Otro prompt creativo
prompt = "A surrealist painting of consciousness exploring the universe, oil on canvas, vivid colors"
print(f"Prompt: {prompt}")

with torch.autocast("cuda"):
    image = pipe(prompt, guidance_scale=7.5).images[0]
display(image)

In [ ]:
del pipe
liberar_memoria()

---
# 19. Visual Question Answering (VQA) con BLIP

Le mostramos una imagen y le hacemos preguntas en lenguaje natural. Combina **vision** y **NLP** en un solo modelo.

**Modelo:** `Salesforce/blip-vqa-base`

In [ ]:
from transformers import BlipProcessor, BlipForQuestionAnswering

vqa_processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
vqa_model = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base")
vqa_model.eval()

img = Image.open("linda_bella.jpg").convert("RGB")
display(img)

for pregunta in ["What animal is in the image?", "How many animals are there?",
                 "What color is the animal?", "Is the animal indoors or outdoors?"]:
    inputs = vqa_processor(img, pregunta, return_tensors="pt")
    with torch.no_grad():
        out = vqa_model.generate(**inputs, max_new_tokens=20)
    print(f"  P: {pregunta}")
    print(f"  R: {vqa_processor.decode(out[0], skip_special_tokens=True)}\n")

In [ ]:
del vqa_model, vqa_processor
liberar_memoria()

---
# Resumen y Conclusiones

Hemos explorado **19 tipos diferentes** de modelos pre-entrenados de deep learning:

### Vision por Computador
| Modelo | Tarea |
|--------|-------|
| ResNet101 | Clasificacion de imagenes |
| DETR | Deteccion de objetos |
| CycleGAN | Transferencia de estilo |
| DeepLabV3 / Mask R-CNN | Segmentacion |
| DPT | Estimacion de profundidad |

### Modelos Multimodales
| Modelo | Tarea |
|--------|-------|
| CLIP | Clasificacion zero-shot |
| BLIP | Descripcion de imagenes |
| BLIP-VQA | Preguntas sobre imagenes |

### Procesamiento de Lenguaje Natural
| Modelo | Tarea |
|--------|-------|
| DistilGPT-2 | Generacion de texto |
| DistilBERT | Analisis de sentimiento |
| BART | Resumen automatico |
| MarianMT | Traduccion automatica |
| DistilBERT-SQuAD | Preguntas y respuestas |
| BERT-NER | Reconocimiento de entidades |
| BERT | Fill-mask |

### Audio
| Modelo | Tarea |
|--------|-------|
| Whisper | Transcripcion de voz |
| AST | Clasificacion de audio |
| MusicGen | Generacion de musica |

### Generativo
| Modelo | Tarea |
|--------|-------|
| Stable Diffusion | Generacion de imagenes |

---

**Recursos para seguir aprendiendo:**
- Hugging Face Hub - Miles de modelos pre-entrenados
- PyTorch Tutorials - Tutoriales oficiales de PyTorch
- Papers With Code - Papers con implementaciones

> Todos estos modelos son solo la *punta del iceberg*. El ecosistema de deep learning crece cada dia. Cambia los inputs, prueba otros modelos, rompe cosas!